# Four-class SPI evidence campaign — 20261027_spi_evidence_v3b

Repository: SemanticSchedulerEnd2End. Immutable source snapshot embedded and
verified; commit alone does not identify the dirty source. Notebook imports the
shared parameters package; no source fork is edited in Kaggle.

Existing identity: satyapaladugu/betterscheduler-spi-20261027finalc; publish a new version, not a new notebook.
Input: satyapaladugu/betterscheduler-spiv2-inputs-20261027b, up to 300 exposed development images and cached CLIP features.
No independent natural-image SPI gold is supplied. All visual labels generated
here are pseudo-labels. Authored text controls measure rubric conformance only.

Managed Kaggle runtime, not the local .venv. Setup pins/reports actual libraries.
Requires two T4s; GPU1 loads one NF4 teacher with thinking disabled; GPU0 stays
unused because features are cached. Both GPUs count toward quota. Maximum
server 7200s = 4 aggregate GPU-hours.

Stages: input/preflight; two prompt protocols on 48 fixed text controls; bounded
challenger only if needed; image labels; all-four-class support gate; CPU grouped
linear/MLP selection with separate early stopping; model export and metrics.

Outputs: /kaggle/working/experiments/spi_four_class/runs/20261027_spi_evidence_v3b/ and
/kaggle/working/run_recovery.zip. Partial labels/raw requests/status are saved
incrementally and campaign recovery runs on failure. Cell logs and an execution
notebook are saved by CellRecorder; verify remote persistence and retrieval.

Teacher controls do not prove visual urgency accuracy. No class is invented,
removed, or relabeled to fill a quota. A failed gate preserves evidence and stops
the corresponding downstream stage. No scheduler policy or codec is trained.


In [ ]:
CELL_SOURCES=["import os,sys,json,base64,zipfile,io,hashlib,time\nfrom pathlib import Path\nprint('START SPI v3 immutable source setup',flush=True)\nROOT=Path('/kaggle/working/spi_source_20261027_spi_evidence_v3b')\nROOT.mkdir(exist_ok=False)\nwith zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as archive:\n    if any(not (ROOT/name).resolve().is_relative_to(ROOT) for name in archive.namelist()):raise ValueError('unsafe source archive')\n    archive.extractall(ROOT)\nSOURCE=json.loads((ROOT/'SOURCE_MANIFEST.json').read_text())\nfor name,expected in SOURCE['files'].items():\n    if hashlib.sha256((ROOT/name).read_bytes()).hexdigest()!=expected:raise ValueError('source checksum mismatch')\nsys.path.insert(0,str(ROOT))\nos.environ['BETTER_SOURCE_MANIFEST']=str(ROOT/'SOURCE_MANIFEST.json')\nos.environ['HF_HOME']='/kaggle/temp/spi_models'\nfor key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):os.environ[key]='1'\nfrom parameters.cloud_notebook import CellRecorder\nrecorder=CellRecorder(CELL_SOURCES,Path('/kaggle/working/experiments/spi_four_class/runs/20261027_spi_evidence_v3b/cell_evidence'))\nwith recorder.cell(0):\n    print('Source',SOURCE['commit'],'files',len(SOURCE['files']),flush=True)\n    print('Interpreter',sys.executable,sys.version,flush=True)\n    print('Thinking OFF; one GPU1 quantized teacher at a time; cached CLIP; no YOLO or scheduler training',flush=True)\n", "with recorder.cell(1):\n    import subprocess,importlib.metadata as metadata,importlib.util\n    import torch\n    print('START dependency/hardware preflight; retain managed Torch/CUDA',flush=True)\n    if torch.cuda.device_count()!=2:raise RuntimeError('existing notebook contract requires two GPUs')\n    print('GPUs',[torch.cuda.get_device_name(i) for i in (0,1)],flush=True)\n    result=subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',\n        'transformers==5.3.0','huggingface-hub==1.5.0','accelerate==1.12.0',\n        'bitsandbytes==0.49.2','sentencepiece==0.2.2','scikit-learn==1.7.2',\n        'nbformat==5.10.4'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=360)\n    print(result.stdout,flush=True);result.check_returncode()\n    from transformers import Qwen3_5ForConditionalGeneration,Qwen3VLForConditionalGeneration\n    if any(importlib.util.find_spec(x) is not None for x in ('fla','causal_conv1d')):\n        raise RuntimeError('unverified optional DeltaNet kernels inherited')\n    from parameters.artifacts import write_json\n    write_json(Path('/kaggle/working/runtime.json'),{'python':sys.version,'interpreter':sys.executable,\n        'packages':{p:metadata.version(p) for p in ('torch','numpy','transformers','huggingface-hub','accelerate','bitsandbytes','scikit-learn','nbformat')},\n        'gpus':[torch.cuda.get_device_name(i) for i in (0,1)],'cuda':torch.version.cuda,'thinking':False})\n    import shutil\n    shutil.copy2('/kaggle/working/runtime.json',recorder.output.parent/'runtime.json')\n    print('END dependency/hardware preflight',flush=True)\n", "with recorder.cell(2):\n    import signal\n    from parameters.cloud_notebook import locate\n    from parameters.spi_pipeline import execute\n    def cutoff(signum,frame):raise TimeoutError('whole-cell budget exceeded')\n    signal.signal(signal.SIGALRM,cutoff);signal.alarm(6720)\n    INPUT_ROOT=locate('input_manifest.json').parent\n    OUTPUT=execute(INPUT_ROOT,Path('/kaggle/working/experiments/spi_four_class/runs/20261027_spi_evidence_v3b/campaign'),\n        seconds=6660,maximum_images=300,runtime_receipt=recorder.output.parent/'runtime.json')\n    signal.alarm(0)\n    print('ACTUAL SPI OUTCOME',json.loads((OUTPUT/'status.json').read_text()),flush=True)\n", "with recorder.cell(3):\n    import zipfile\n    run=Path('/kaggle/working/experiments/spi_four_class/runs/20261027_spi_evidence_v3b')\n    files=[p for p in run.rglob('*') if p.is_file() and '/data/images/' not in str(p) and p.name!='run_recovery.zip']\n    from parameters.artifacts import write_json\n    write_json(run/'retrieval_checksums.json',{str(p.relative_to(run)):hashlib.sha256(p.read_bytes()).hexdigest() for p in files\n        if 'cell_evidence' not in str(p)})\n    with zipfile.ZipFile('/kaggle/working/run_recovery.zip','w',zipfile.ZIP_DEFLATED) as archive:\n        for p in files:archive.write(p,str(p.relative_to(run)))\n        archive.write(run/'retrieval_checksums.json','retrieval_checksums.json')\n    print('END recovery export; inspect campaign/status.json and verify checksums on retrieval',flush=True)\n"]
import os,sys,json,base64,zipfile,io,hashlib,time
from pathlib import Path
print('START SPI v3 immutable source setup',flush=True)
ROOT=Path('/kaggle/working/spi_source_20261027_spi_evidence_v3b')
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as archive:
    if any(not (ROOT/name).resolve().is_relative_to(ROOT) for name in archive.namelist()):raise ValueError('unsafe source archive')
    archive.extractall(ROOT)
SOURCE=json.loads((ROOT/'SOURCE_MANIFEST.json').read_text())
for name,expected in SOURCE['files'].items():
    if hashlib.sha256((ROOT/name).read_bytes()).hexdigest()!=expected:raise ValueError('source checksum mismatch')
sys.path.insert(0,str(ROOT))
os.environ['BETTER_SOURCE_MANIFEST']=str(ROOT/'SOURCE_MANIFEST.json')
os.environ['HF_HOME']='/kaggle/temp/spi_models'
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):os.environ[key]='1'
from parameters.cloud_notebook import CellRecorder
recorder=CellRecorder(CELL_SOURCES,Path('/kaggle/working/experiments/spi_four_class/runs/20261027_spi_evidence_v3b/cell_evidence'))
with recorder.cell(0):
    print('Source',SOURCE['commit'],'files',len(SOURCE['files']),flush=True)
    print('Interpreter',sys.executable,sys.version,flush=True)
    print('Thinking OFF; one GPU1 quantized teacher at a time; cached CLIP; no YOLO or scheduler training',flush=True)


In [ ]:
with recorder.cell(1):
    import subprocess,importlib.metadata as metadata,importlib.util
    import torch
    print('START dependency/hardware preflight; retain managed Torch/CUDA',flush=True)
    if torch.cuda.device_count()!=2:raise RuntimeError('existing notebook contract requires two GPUs')
    print('GPUs',[torch.cuda.get_device_name(i) for i in (0,1)],flush=True)
    result=subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
        'transformers==5.3.0','huggingface-hub==1.5.0','accelerate==1.12.0',
        'bitsandbytes==0.49.2','sentencepiece==0.2.2','scikit-learn==1.7.2',
        'nbformat==5.10.4'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=360)
    print(result.stdout,flush=True);result.check_returncode()
    from transformers import Qwen3_5ForConditionalGeneration,Qwen3VLForConditionalGeneration
    if any(importlib.util.find_spec(x) is not None for x in ('fla','causal_conv1d')):
        raise RuntimeError('unverified optional DeltaNet kernels inherited')
    from parameters.artifacts import write_json
    write_json(Path('/kaggle/working/runtime.json'),{'python':sys.version,'interpreter':sys.executable,
        'packages':{p:metadata.version(p) for p in ('torch','numpy','transformers','huggingface-hub','accelerate','bitsandbytes','scikit-learn','nbformat')},
        'gpus':[torch.cuda.get_device_name(i) for i in (0,1)],'cuda':torch.version.cuda,'thinking':False})
    import shutil
    shutil.copy2('/kaggle/working/runtime.json',recorder.output.parent/'runtime.json')
    print('END dependency/hardware preflight',flush=True)


In [ ]:
with recorder.cell(2):
    import signal
    from parameters.cloud_notebook import locate
    from parameters.spi_pipeline import execute
    def cutoff(signum,frame):raise TimeoutError('whole-cell budget exceeded')
    signal.signal(signal.SIGALRM,cutoff);signal.alarm(6720)
    INPUT_ROOT=locate('input_manifest.json').parent
    OUTPUT=execute(INPUT_ROOT,Path('/kaggle/working/experiments/spi_four_class/runs/20261027_spi_evidence_v3b/campaign'),
        seconds=6660,maximum_images=300,runtime_receipt=recorder.output.parent/'runtime.json')
    signal.alarm(0)
    print('ACTUAL SPI OUTCOME',json.loads((OUTPUT/'status.json').read_text()),flush=True)


In [ ]:
with recorder.cell(3):
    import zipfile
    run=Path('/kaggle/working/experiments/spi_four_class/runs/20261027_spi_evidence_v3b')
    files=[p for p in run.rglob('*') if p.is_file() and '/data/images/' not in str(p) and p.name!='run_recovery.zip']
    from parameters.artifacts import write_json
    write_json(run/'retrieval_checksums.json',{str(p.relative_to(run)):hashlib.sha256(p.read_bytes()).hexdigest() for p in files
        if 'cell_evidence' not in str(p)})
    with zipfile.ZipFile('/kaggle/working/run_recovery.zip','w',zipfile.ZIP_DEFLATED) as archive:
        for p in files:archive.write(p,str(p.relative_to(run)))
        archive.write(run/'retrieval_checksums.json','retrieval_checksums.json')
    print('END recovery export; inspect campaign/status.json and verify checksums on retrieval',flush=True)
